# Instalacion de las librerias necesarias

In [1]:
!pip install gym stable-baselines3 matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.2/187.2 kB 11.2 MB/s eta 0:00:00


In [2]:
!pip install shimmy>=2.0

se instala "shimmy"Para que no haya errores de compatibilidad mientra se usa la libreia de Gym con la libreria de stable_baselines3

# Librerias

In [3]:
import numpy as np
import gym
from gym import spaces #define los espacios y las acciones
from stable_baselines3 import PPO
import numpy as np
import time
import plotly.graph_objects as go

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=

Se crea una grilla 7x7 con:

      - Meta (🏡) en la celda 48

      - Semáforos (🚥) en las celdas 10, 27, 30, 42 (recompensa negativa)

      - Túneles (🌉): Se encuentre en la celda 3 que teletransportaria al agente a la celda 28 y en la celda 36 a la celda 20

      - El agente debe recoger a una Persona (👷🏽) antes de ir a la Meta.

In [4]:
class Entorno_del_Agente(gym.Env):
  # Este diccionario le dice a Gym que el entorno solo puede renderizarse en modo texto, gracias a la calve de "render.modes"
    metadata = {'render.modes': ['console']}

    #el agente trabaja con acciones como valores numéricos
    ARRIBA, IZQUIERDA, ABAJO, DERECHA = 0, 1, 2, 3

    def __init__ (self, persona_pos: int =8): #persona_pos posicion de la persona a buscar
      super().__init__() #Se usa 'super' para poder llamar a la clase 'gym.Env' para inicarlo correctamente
      self.celda = 7
      self.celdas = self.celda * self.celda

      # Configuración fija de los atributos
      self.meta = 48
      self.semaforos = {10, 27, 30, 42}
      self.tuneles = {3: 28, 36: 20}
      self.persona_pos_inicial  = int(persona_pos)

      #cantidad de moviminetos que puede hacer el agente (define el espacio de acciones)
      self.action_space = spaces.Discrete(4)
      # solo define que hay 98 estados posibles.
      self.observation_space = spaces.Discrete(self.celdas * 2)

      # Recompensas
      self.pasos = -1
      self.penalizacion_semaforo = -5
      self.recompensa_meta = 50
      self.sinpasajero = -5
      self.conpasajero = 10

      #Estados del entorno
      self.posicion_agente = None #esta variable guarda el índice de la celda actual del agente (taxi)
      self.pasajero = False #indica si el agente ya recogió o no al pasajero
      self.persona_pos = None #guarda la posición actual de la persona
      self.reset() #esta línea reinicia el entorno para empezar una nueva partida.

    # utilidades internas para que el entorno pueda validar la información del estado
    def _fila_col(self, estado: int):
      fila = estado // self.celda #Eso determina la fila actual del agente en la grilla. (Descarte el decimal)
      columna = estado % self.celda #Eso determina la columna actual del agente en la grilla (usando el resto)
      return fila, columna

      # determinamos si los valores de las filas y las columnas estan dentro de lo habitual
    def _dentro (self, fila: int, col: int)-> bool:
      if (fila >= 0) and (fila < self.celda):
        dentro_fila = True
      else:
        dentro_fila = False

      if (col >= 0) and (col < self.celda):
        dentro_columna = True
      else:
        dentro_columna = False

      return dentro_fila and dentro_columna

    #Esa funcion sirve para indicar si el agente recogio al pasajero o no, en donde el agente se va a encontrar en el mismo indice de la celda, pero el numero de la celda (estado) v a ser diferente
    def _encode_obs (self) -> int:
      return self.posicion_agente + self.celdas * int(self.pasajero) #se trnaforma el booleano en numero

    #reindica la posicion de la persona una vez que termino el episodio
    def reset(self): #gym
      self.persona_pos = self.persona_pos_inicial
      self.pasajero = False #indica que el agente no recogio
    # Estados no válidos para iniciar
      no_inicio = {self.meta, self.persona_pos} | self.semaforos | set(self.tuneles.keys()) #solo obtengo las llaves
      agente = [s for s in range(self.celdas) if s not in no_inicio]
      self.posicion_agente = int(np.random.choice(agente))
      return self._encode_obs()

    def step(self, action: int): #gym
      fila, col = self._fila_col(self.posicion_agente)
      if action == self.ARRIBA:
          nf, nc = fila -1, col #si el agente esta en la fila 4 y quiere subir a la fila 3, se resta la fila 4 para que llegue a fila 3
      elif action == self.IZQUIERDA:
        nf, nc = fila, col -1
      elif action == self.ABAJO:
        nf, nc = fila +1, col
      elif action == self.DERECHA:
          nf, nc = fila, col+1
      else:
          raise ValueError(f"Accion inavlida: {action}")

      if self._dentro(nf, nc): # se verifica que los valores de la columna y de la fila esten dentro del rango permitido
          self.posicion_agente = nf * self.celda + nc #Se obtiene la posicion exacte del agente en la grilla
      # Túneles
      if self.posicion_agente in self.tuneles:
          self.posicion_agente = self.tuneles[self.posicion_agente] #mueve al agente desde la entrada del túnel hacia la salida correspondiente.

      reward = self.pasos
      done = False #Variable que le dice al agente cuando se termino el episodio

        #Recompensa al subir al pasajero
      if (self.persona_pos is not None) and (self.posicion_agente == self.persona_pos):
          self.pasajero = True
          self.persona_pos = None # ya no se muestra en el mapa
          reward += self.conpasajero

        #Penalizacion de los semaforos
      if self.posicion_agente in self.semaforos:
          reward += self.penalizacion_semaforo

        #Meta
      if self.posicion_agente == self.meta:
        if self.pasajero:
            reward += self.recompensa_meta #La recompensa se guarda gracias al "+="
        else:
            reward += self.sinpasajero - 20
        done = True
      info = {} #valor que devuelve la libreria de GYM
      return self._encode_obs(), reward, done, info

    def close(self): #gym
      pass #no hay nada para cerrar

# Modelo PPO

In [ ]:
entorno = Entorno_del_Agente()
agente = PPO("MlpPolicy", entorno,learning_rate=0.0004, clip_range=0.15, verbose=1) #verbose muestra la informacion del entrenamiento del agente

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


El modelo va a presentar un aprendizaje levemente más rápido (debido al aumento del valor de learning_rate, ya que el valor por defecto es de 0.0003), pero será menos tolerante a los cambios de política, clip range Verfica el ratio (1) puede variar entre hasta un 15% del valor de ratio (es decir que el ratio seria entre 0.85, 1.15) en la que el modelo tome una accion, si esa misma accion en el mismo estado supera el umbral del 15% (0.15) es decir el ratio mide cuánto cambia la probabilidad de elegir esa acción entre política nueva y la vieja. En donde si el valor del ratio es del (1.35) lo recorte y le asgina el valor de 1.15, si o el valor del ratio es del (0.35) lo recorte y le asgina el valor de 0.85.

##` Arquitectura del modelo PPO `

In [ ]:
print(agente.policy)

ActorCriticPolicy(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (pi_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (vf_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (mlp_extractor): MlpExtractor(
    (policy_net): Sequential(
      (0): Linear(in_features=98, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
    (value_net): Sequential(
      (0): Linear(in_features=98, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
  )
  (action_net): Linear(in_features=64, out_features=4, bias=True)
  (value_net): Linear(in_features=64, out_features=1, bias=True)
)


El modelo de PPO es un modelo con la estructura de arquitectura actor-crítico. Presenta 2 arquitecturas de red iguales pero con diferentes funciones:


La primera capa ”FlattenExtractor" es la encargada de recibir cualquier estructura de observacion (imagnes, matrices etc) y las transforma en un vector unidimensional, en este caso la capa "”FlattenExtractor" no se usa por que tenga 2 capas de extractor por seaprado.
Las capas de "pi_features_extractor" y "vf_features_extractor", cumplen exactamente la misma función que “FlattenExtractor“, las 2 capas reciben las misma observaciones, en donde la capa de "pi_features_extractor", es la encargada de recuperar información para la toma de acciones del modelo y la capa de "vf_features_extractor" es la encargada de poder extraer información para la red densa encargada de predecir/asignar el valor del estado.


Como se puede observar el modelo presentan 2 arquitecturas de red Linear (es decir que tiene una capa de red densa, hidden layers)

 - **La capa de “policy_net” (Actor):** Obtiene las features del extractor de “pi_features_extractor", es la encargada de predecir qué acciones va a tomar el agente. En donde recibe las 98 features de entrada y las reduce a 64 nueronas de salidas (cada neurona procesa las 98 feature), que luego los pesos (w) y el bias pasan por la funcion de activacion hiperbolica tangente, luego se repite el mismo proceso para la segunda capa que tiene como entrada 64 features y como salida 64 neuronas.

 - **La capa de “value_net” (crtico):** Obtiene la caracteristicas del extractor de “vf_features_extractor", es la encargada de predecir el valor del estado. Tiene el mismo funcionamiento que la capa de “policy_net”.


Se debe de tener en cuenta que en ambas capas se aplica una reducción de dimensionalidad con el objetivo de poder:
 - Aprender una representación compacta: La red aprende a codificar la información esencial del estado en menos dimensiones.
 - Mejorar la generalización: Reducir dimensionalidad fuerza a la red a encontrar patrones generales en lugar de memorizar combinaciones exactas de features ayudando al agente a aprender políticas más robustas
 - Facilitar el cálculo del valor del estado y de las acciones



Por último se pasa 2 capas finales:
 - **La capa de action_net:** Es la encargada de devolver 4 posibles acciones a tomar, este capa devuelve 4 valores en crudo llamados logits (es la puntuacion que el modelo le da a las acciones en el estado en el que se encuentra, es decir que predice que tan buena considera la accion el modelo en el estado) se convierten en una distribución de probabilidad categórica. Durante entrenamiento se muestrea (por ejemplo: Si ARRIBA tiene 70%, se elegirá el 70% de las veces, pero a veces también se eligen las otras acciones) de esta distribución, luego con con deterministic=True se elige la accion con la probabildad mas alta.
 - **La capa de (value_net)** es la encargada de devolver el valor del estado

## Entrenaminento del modelo PPO

In [ ]:
agente.learn(total_timesteps=100000) # cantidad de pasos totales que tiene que realizar el agente para poder completar el entrenamiento
agente.save("agente_tuning")

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 133      |
|    ep_rew_mean     | -187     |
| time/              |          |
|    fps             | 788      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 154          |
|    ep_rew_mean          | -209         |
| time/                   |              |
|    fps                  | 569          |
|    iterations           | 2            |
|    time_elapsed         | 7            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0007158275 |
|    clip_fraction        | 0.000537     |
|    clip_range           | 0.15         |
|    entropy_loss         | -1.39        |
|    explained_variance   | -3.39e-05    |
|    learning_r

##` Explicacion de las metricas`

- **Rollout**: Valores que se calculan mediante la interaccion del agente con el entorno
     - ep_lean_mean: Es el promedio de los pasos que el agente tardo en terminar el episodio, teniendo en cuenta todos los episodios ejecutados hasta el momento.
     - ep_row_mean: promedio de las recompensa recibidas del agente en el episodio, teniendo en cuenta todos los episodios ejecutados hasta el momento.

- **Time** : Valores del rendimiento computacional:

   - Fps: Cantidad de pasos procesados por segundos, valores mas alto = entrenamiento mas rapido.
   - Iterations: Cantidad de iteraciones completadas (Cuántas veces el agente realizo el proceso de recolectar datos y lanzó un ciclo de entrenamiento, es decir que entreno la red. En donde una vez completdo el ciclo recien se cuenta como una iteracion).
   - time_elapsed: tiempo total de entrenamiento en segundos.
   - total_timesteps: Cantidad total de pasos.


- **Train**: Proceso de optimizacion de la red.
   
   -  aprox_kl (Divergencia de Kullback-Leiber): Sirve para saber cuanto cambio la politica anterior respecto a la nueva politica del agente, Valores bajos Politica similar a la anterior (solo de la poltica anterior, no de todas), significa aprendizaje estable y valores altos significa que la nueva política cambió demasiado (riesgo de inestabilidad o pérdida de lo aprendido).

   - clip_fraction: Indica el porcentaje de las actulzaciones de la politca que fueron recortadas por el clip_range en la ultima actualizacion de politicas, en donde valores cerca del 1 significa que el clip_range este interviniendo mucho en la actulzacion de la politica y valores cerca del 0 significa que el clip_range no esta interviniendo tanto en la actulizacion de la politicas

   - clip_range: Define el límite máximo de cambio (del ratio) en la que el agente tome la misma accion entre la politca nueva y la vieja (por defecto 0.2).

   - entropy_loss: Mide el grado de aleatoriedad de la política. Cuanto más negativo, menos exploración (más determinista, es decir que el agente empieza a elegir consistentemente las acciones que sabe que le dan buena recompensa, tambien se lo suele llamar "Explotacion").

   el signo negativo aparece en "entropy_loss", por que se intenta maximizar la exploracion pero como los optimizadores (Adam por ejemplo) estan diseñados para poder minimizar la funcion de costo, se le asigna el signo con un valor negativo, en donde en cada iteraccion si el valor disminuye, como por ejemplo:
   entropy_loss| -0.014
   entropy_loss| -0.0193
   Significa que el agente esta siendo mas determinista (se redece la entropia)
   En cambio si aumenta:
     entropy_loss| -0.0193
     entropy_loss| -0.014
  Significa que el agente esta maximizando la exploracion.

   - explained_variance: Evalua las predicciones del critico mediante la varianza, con respecto a los valores de estados reales, en donde: Si el valor es cercano o igual a 1 el critico predice muy bien las recomepnsas (excelente aprendizaje) valores cercanos o iguales a 0 el critico no aprendio nada del entorno, si el valor es negativo predice valores altos cuando en realidad los valores reales son bajas, y viceversa.

   - learning_rate: tasa de aprendizaje actual.

   - loss → pérdida total de entrenamiento combinada entre:

     policy_gradient_loss → mide qué tan bien mejora la política del actor (las acciones).

     value_loss → mide qué tan bien el crítico (Value_net) predice las recompensas.

     entropy_loss →  Mide el grado de aleatoriedad de la política (Es decir mide sie el agente esta siendo mas determinista o si esta priorizando la exploracion)

   - n_updates: Hace referencia a la cantidad total de veces en el que el agente actualizo los pesos de la red teniendo en cuenta todas las iteraciones.

   - policy_gradient_loss: Pérdida asociada al gradiente de la red "policy_net" (actor)
   El signo negativo aparece, por que se intenta maximizar la recompensa pero como los optimizadores (Adam por ejemplo) estan diseñados para poder minimizar la funcion de costo, se le asigna el signo del valor negativo, en donde en cada iteraccion si el valor disminuye, como por ejemplo:
   policy_gradient_loss |-0.014 policy_gradient_loss | -0.0193
   Significa que el agente esta maximizando las recompensas (siendo mas determinista), ya que significa que la funcion de costo esta direccionandose a un minimo local lo que en consecuencia le permite al agente maximizar las recompensas

   - El value_loss: Es la pérdida (error) de la red del crítico (Value_net) al tratar de predecir el valor real de los estados, en donde mide el error cuadrático medio (MSE) entre lo que el crítico predice y el valor de estado real


   Los valores presentados por: value_loss, policy_gradient_loss, loss y entropy_loss siempre son en relacion a la ultima iteracion

# Testing

In [ ]:
N_EPISODIOS = 1000
RUTA_MODELO = "agente_tuning"

# Cargar modelo y crear entorno
modelo = PPO.load(RUTA_MODELO)
entorno = Entorno_del_Agente()

exitos_con_persona = 0           # llegó a meta habiendo recogido a la persona
llegadas_sin_persona = 0         # llegó a meta sin persona
pasos_por_epi = []

for epi in range(1, N_EPISODIOS + 1): # se coloca 1 para que empiece contando desde 1 y se agrega un '+1' para asegurarse a que llegue a los 1000 episodios
    obs = entorno.reset()
    done = False
    pasos = 0

    while not done:
        accion, _ = modelo.predict(obs, deterministic=True) #Siempre devuelve accion y estado y obs se actualiza con cada iteraccion del paso del agente
        obs, recompensa, done, info = entorno.step(accion) #Cada ejecuccion de la funcion 'step' es un paso del agente
        pasos += 1

    # Al terminar el episodio, distinguimos el motivo de finalización
    if entorno.posicion_agente == entorno.meta:
        if entorno.pasajero:          # terminó en meta y con persona
            exitos_con_persona += 1
        else:                          # terminó en meta pero sin persona
            llegadas_sin_persona += 1

    pasos_por_epi.append(pasos)

promedio_pasos = float(np.mean(pasos_por_epi))
epi_max_pasos = int(np.argmax(pasos_por_epi))
epi_min_pasos = int(np.argmin(pasos_por_epi))


print("===== RESUMEN (AGENTE ENTRENADO) =====")
print(f"Llegó a la meta CON persona: {exitos_con_persona} de {N_EPISODIOS}")
print(f"Llegó a la meta SIN persona: {llegadas_sin_persona} de {N_EPISODIOS}")
print(f"Promedio de pasos por episodio: {promedio_pasos:.2f}")
print(f"Episodio con MÁS pasos: {pasos_por_epi[epi_max_pasos]} pasos")
print(f"Episodio con MENOS pasos: {pasos_por_epi[epi_min_pasos]} pasos")

===== RESUMEN (AGENTE ENTRENADO) =====
Llegó a la meta CON persona: 1000 de 1000
Llegó a la meta SIN persona: 0 de 1000
Promedio de pasos por episodio: 15.05
Episodio con MÁS pasos: 19 pasos
Episodio con MENOS pasos: 11 pasos


In [ ]:
N_EPISODIOS = 1000

# Crear el entorno
entorno = Entorno_del_Agente()

# Métricas
exitos_con_persona = 0        # Llegó a la meta con la persona
llegadas_sin_persona = 0      # Llegó a la meta sin persona
pasos_por_epi = []

for epi in range(1, N_EPISODIOS + 1):
    obs = entorno.reset()
    done = False
    pasos = 0

    while not done:
        # se usa el atributo de 'action_space' y  el metodo de 'sample' de gym para que elija una acción aleatoria.
        accion = entorno.action_space.sample()
        obs, recompensa, done, info = entorno.step(accion)
        pasos += 1

    # Al finalizar el episodio, se verifica cómo terminó
    if entorno.posicion_agente == entorno.meta:
        if entorno.pasajero:
            exitos_con_persona += 1
        else:
            llegadas_sin_persona += 1

    pasos_por_epi.append(pasos)

promedio_pasos = float(np.mean(pasos_por_epi))
epi_max = int(np.argmax(pasos_por_epi))
epi_min = int(np.argmin(pasos_por_epi))

print("===== RESUMEN (AGENTE ALEATORIO) =====")
print(f"Llegó a la meta CON persona: {exitos_con_persona} de {N_EPISODIOS}")
print(f"Llegó a la meta SIN persona: {llegadas_sin_persona} de {N_EPISODIOS}")
print(f"Promedio de pasos por episodio: {promedio_pasos:.2f}")
print(f"Episodio con MÁS pasos: {pasos_por_epi[epi_max]} pasos")
print(f"Episodio con MENOS pasos: {pasos_por_epi[epi_min]} pasos")


===== RESUMEN (AGENTE ALEATORIO) =====
Llegó a la meta CON persona: 482 de 1000
Llegó a la meta SIN persona: 518 de 1000
Promedio de pasos por episodio: 153.15
Episodio con MÁS pasos: 1221 pasos
Episodio con MENOS pasos: 1 pasos


Se observa que el agente entrenado logró alcanzar la meta con el pasajero en las 1.000 ejecuciones realizadas en el experimento, demostrando un comportamiento consistente y óptimo. En contraste, que cuando se elijieron acciones aleatorias, alcanzó la meta en solo 482 ocasiones, evidenciando un desempeño inferior.

Además, el agente entrenado presentó un promedio de 15 pasos por episodio, con un máximo de 19 pasos, lo que refleja una política eficiente y estable para completar la tarea. Por otro lado, el cuando se elijio acciones aleatorias, necesitó en promedio 153 pasos por episodio, con un máximo de 1.221 pasos, lo que indica un comportamiento aleatorio, con trayectorias largas y poco efectivas.

Finalmente, se destaca que cuando se eligio acciones al azar solo se registró un mínimo de 1 paso. Este resultado implica que, en dichos casos, indica que llegó a la meta sin recoger al pasajero, incumpliendo el objetivo del entorno. En cambio, el agente entrenado aprendió a priorizar la recolección del pasajero antes de dirigirse a la meta, cumpliendo correctamente con la consigna planteada.

## Renderizacion con Ploty

In [5]:
""" Ploty, funciona al reves que el entorno, en donde el indice 0 es abajo de todo (no arriba como en el entorno)
    y el indice 6, es arriba de todo (no como en el entorno que es abajo del todo), es por eso que se hace: r = size - 1 - r,
    Se hace la resta, para asi poder restar las filas del indice mayor del (6) desde abajo (como seria en el entorno) por la fila (r) para asi poder encontrar la fila correcta del entorno
    Ejemplo: r=5 (r=7-1-5 = 1) se dibuja en el indice 1, segunda fila del entorno"""

def _idx_to_xy(idx: int, size: int):
    r = idx // size # Se obtiene la fila "Y"
    c = idx % size # Se obtiene la columna "X" (se toma el resto de la división)
    r = size - 1 - r
    return c, r


def _grid_shapes(size: int, cell_size: float = 1.0): #size es el numero de celdas por lado de la grilla (7) y cell_size se usa para calcular tanto el ancho como el alto de la celda.
    rectangulos = [] #lista en donde se guardaran las celdas
    for r in range(size): #recorre los indice de las filas (y)
        for c in range(size): #recorre los indice de las columnas columnas (x)

        #X0 e Y0 sirve para indicar en donde empieza cada celda
            x0 = c * cell_size #indicar en donde empieza la celda de la columna
            y0 = r * cell_size #indicar en donde empieza la celda de la filas
        #X1 e Y1 sirve para indicar en donde termina cada celda y el tamaño de la grilla
            x1 = x0 + cell_size #Delimitador de donde termina la celda de la columnas
            y1 = y0 + cell_size ##Delimitador de donde termina la celda la filas

            rectangulos.append(dict(
                type="rect", #indica que el objeto a dibujar en este caso un rectangulo
                x0=x0, y0=y0, x1=x1, y1=y1, #celdas
                line=dict(color="#B9A6D9", width=1), # devuelve un diccionario con el color y el grosor de la linea del rectangulo (celda)
                fillcolor="#E9DFF8", #color de fondo de la celda.
                layer="below" #se le indica a Ploty que el rectángulo se dibuje debajo de los elementos (en este caso emojis)
            ))


    # líneas de la grilla
    for i in range(size + 1): #Indica la linea la celda
        rectangulos.append(dict(type="line", #le dice a Plotly que el objeto a dibujar es una línea recta
                           #indica el linea vertical, Ploty, espara especifacemente las variables de x0, x1, y0, y1
                           x0=i*cell_size, y0=0, #indica el incio de cada columna (desde el indice 0 de la fila (y0)).
                           x1=i*cell_size, y1=size*cell_size, #indica el final de cada columna (desde el indice 7 de la fila (y1)).
                           line=dict(color="#B9A6D9", width=1))) #Indica el color y el ancho de la linea
                           #(i, siempre va a ser el mismo para x0, x1, para asi poder indicar que esa misma linea va desde el indice 0 de la fila hasta el indice 7 de fila)

        rectangulos.append(dict(type="line",
                           ##indica el linea horizontal
                           x0=0, y0=i*cell_size, #indica el inicio de cada fila (desde el indice 0 de la columna (x0)).
                           x1=size*cell_size, y1=i*cell_size, ##indica el fin de cada fila (desde el indice 7 de la columna (x1)).
                           line=dict(color="#B9A6D9", width=1)))
    return rectangulos #Se Obtiene una lista de diccionarios, que tiene anidados diccionarios con las cantidades de celdas, con el color grosor del linea del rectángulo, color de fondos de celda etc.

def _annot(x, y, text, cell_size=1.0): #Text=emoji
    """Crea una anotación centrada en la celda (x,y)."""
    return dict(
        #Coordenadas que se le pasa a Ploty para que sepa en donde se dibuja el Emoji
        x=x*cell_size + cell_size/2, #se coloca el emoji al centro de la celda (horizontal, ancho de la celda ya que x es la columna)
        y=y*cell_size + cell_size/2, #se coloca el emoji al centro de la celda (vertical, alto de la celda ya que y es la fila )
        text=text, #emoji
        showarrow=False, #no dibuje una flecha apuntando al emoji (asi esta configurado Ploty)
        font=dict(size=22), #Define el tamaño de la fuente del emoji. (Ploty ya maneja el emoji en la variable text y solo le aplica el tamaño de la fuente seleccionado)
        xanchor="center", #centra el emoji horizontalmente.
        yanchor="middle" #centra el emoji verticalmente.
    )

def render_plotly(env, show=True): #show = Ture para poder mostrar la figura en pantalla
    """Renderiza el estado actual del entorno Entorno_del_Agente con Plotly."""
    size = env.celda
    cell = 1.0

    fig = go.Figure() #Se crea un lienzo
    fig.update_layout(
        width=520, height=520, #define el tamaño de la figura
        margin=dict(l=10, r=10, t=10, b=10), #establece los tamaños de los márgenes
        xaxis=dict(visible=False, range=[0, size]), #oculta el eje X (no se ven los números ni la línea del eje de Ploty) y le dice a Ploty que el eje de las x va a tener 7 indices.
        yaxis=dict(visible=False, range=[0, size], scaleanchor="x", scaleratio=1), #oculta el eje Y (no se ven los números ni la línea del eje propio de Ploty) y le dice a Ploty que el eje de las x va a tener 7 indices, con "scaleanchor" se asegura que la canitdad de pixeles usados para el alto de la celda (eje Y) sea el mismo que el del ancho de la celda (eje X), de acuuerdo al espacio asignado por "scaleratio" y "scaleratio" nos asegurames que el espacio dentro de la celda de alto del eje (1) y sea el mismo que el espacio de ancho del eje x (1), es decir que ambos ejes tendrian la misma distancia de espacios dentro de la celda.
        shapes=_grid_shapes(size=size, cell_size=cell) #Se crea la grilla
    )

    lista = []

    # Meta (🏡)
    x, y = _idx_to_xy(env.meta, size) #Se obtiene fila y columnas
    lista.append(_annot(x, y, "🏡", cell)) #Se centraliza el emoji y "cell" reemplaza a cell_size

    # Semáforos (🚥)
    for s in env.semaforos:
        x, y = _idx_to_xy(s, size)
        lista.append(_annot(x, y, "🚥", cell))

    # Túneles (🌉)
    for t_in in env.tuneles.keys():
        x, y = _idx_to_xy(t_in, size)
        lista.append(_annot(x, y, "🌉", cell))

    # Persona (👷🏽)
    if env.persona_pos is not None:
        x, y = _idx_to_xy(env.persona_pos, size)
        lista.append(_annot(x, y, "👷🏽", cell))

    # Agente (🚕)
    xa, ya = _idx_to_xy(env.posicion_agente, size)
    lista.append(_annot(xa, ya, "🚕", cell))

    fig.update_layout(annotations=lista) #agrega todas las anotaciones (emojis) de la lista al gráfico
    #para las celdas que no tienen emoji solo tienen el color definido de la celda

    if show:
        fig.show() #se imprime el lienzo con la grilla hecha (figura)
    return fig

In [13]:
#Cargar el agente entrenado
#modelo = PPO.load("agente_tuning")

#Crear el entorno
entorno = Entorno_del_Agente()

# Reiniciar el entorno para obtener el estado inicial
observacion = entorno.reset()
terminado = False

#Ejecutar un episodio completo y renderizar con Plotly
while not terminado:
    # Predecir la acción según la política entrenada
    accion, _estados = modelo.predict(observacion, deterministic=True) #deterministic=True su usa para que el modelo elige siempre la acción más probable según su política

    # Ejecutar la acción en el entorno
    observacion, recompensa, terminado, info = entorno.step(accion) #Se ultiliza el metodo 'step' de la libreria de gym para que se pueda ejecutar la accion en el entorno

    # Renderizar visualmente con Plotly
    render_plotly(entorno)

    # Pausa pequeña para visualizar los pasos del agente (el agente, ejcuta una accion se para el entorno por 0.5 y se vuelve a ejecutar otra accion)
    time.sleep(0.5)

print("Episodio finalizado.")

Episodio finalizado.


# Se guarda el modelo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
modelo.save("/content/drive/MyDrive/agente")

Mounted at /content/drive


In [ ]:
modelo = PPO.load("/content/drive/MyDrive/agente")